In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest

# Load the dataset
def load_data(file_path):
    df = pd.read_csv(file_path)
    numeric_cols = ['Has_Response', 'Vibe_Score', 'Has_Award', 'Has_Activity', 
                    'Teams_Messages_Sent', 'Emails_Sent', 'Meetings_Attended', 
                    'Work_Hours', 'Leave_Days', 'Promotion_Consideration', 'Performance_Rating']
    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')
    return df

# Data Preprocessing
def preprocess_data(df):
    df['Total_Communications'] = df['Teams_Messages_Sent'] + df['Emails_Sent'] + df['Meetings_Attended']
    df['Activity_Performance_Ratio'] = df.apply(
        lambda row: row['Total_Communications'] / (row['Performance_Rating'] + 0.1) 
        if row['Total_Communications'] > 0 else 0, axis=1
    )
    avg_leave = df['Leave_Days'].mean()
    df['Leave_Anomaly'] = abs(df['Leave_Days'] - avg_leave) / avg_leave
    df['Is_Inactive'] = ((df['Total_Communications']) == 0).astype(int)
    return df

# Calculate risk dimensions
def calculate_risk_dimensions(df):
    df['Engagement_Deficit'] = 10 - (df['Vibe_Score'] * 5) - (df['Has_Response'] * 1.5) - (df['Has_Activity'] * 2)
    df['Engagement_Deficit'] = df['Engagement_Deficit'].clip(0, 10)

    df['Effort_Score'] = (df['Work_Hours']/30) * 5 + (df['Total_Communications']/150) * 5
    df['Performance_Effort_Misalignment'] = (df['Effort_Score'] - (df['Performance_Rating'] * 2.5)).clip(0, 10)

    df['Expected_Awards'] = df['Performance_Rating'] * 0.7
    df['Recognition_Imbalance'] = (5 * (df['Expected_Awards'] - df['Has_Award']).clip(0) + 
                                   5 * (1 - df['Promotion_Consideration'])).clip(0, 10)

    df['Work_Hours_Z'] = (df['Work_Hours'] - df['Work_Hours'].mean()) / df['Work_Hours'].std()
    df['Work_Pattern_Anomalies'] = ((abs(df['Leave_Anomaly']) * 5) + (abs(df['Work_Hours_Z']) * 2)).clip(0, 10)

    df['Career_Progression_Stagnation'] = (1 - df['Promotion_Consideration']) * 10
    return df

# Detect anomalies using Isolation Forest
def detect_anomalies(df):
    features = ['Vibe_Score', 'Has_Response', 'Total_Communications', 'Work_Hours', 'Leave_Days', 'Performance_Rating', 'Promotion_Consideration']
    X = df[features].fillna(0)
    if len(df) > 10:
        model = IsolationForest(contamination=0.1, random_state=42)
        df['Anomaly_Score'] = model.fit_predict(X)
        df['Anomaly_Score'] = df['Anomaly_Score'].map({1: 0, -1: 1})
    else:
        df['Anomaly_Score'] = 0
    return df

# Peer group comparisons
def create_peer_comparisons(df):
    df['Activity_Level'], bins = pd.qcut(df['Total_Communications'].clip(0, 200), q=4, labels=False, duplicates='drop', retbins=True)
    labels = {2:['Low','High'], 3:['Low','Medium','High'],4:['Low','Medium-Low','Medium-High','High']}[len(bins)-1]
    df['Activity_Level'] = pd.Categorical.from_codes(df['Activity_Level'], labels, ordered=True)

    for metric in ['Vibe_Score', 'Performance_Rating', 'Work_Hours']:
        df[f'{metric}_Z'] = df.groupby('Activity_Level', observed=True)[metric].transform(lambda x: (x - x.mean()) / (x.std() if x.std() > 0 else 1))

    df['Peer_Deviation'] = (abs(df['Vibe_Score_Z']) + abs(df['Performance_Rating_Z']) + abs(df['Work_Hours_Z'])) / 3
    return df

# Overall risk scoring
def calculate_risk_score(df):
    weights = {'Engagement_Deficit':0.3,'Performance_Effort_Misalignment':0.25,'Recognition_Imbalance':0.15,'Work_Pattern_Anomalies':0.15,'Career_Progression_Stagnation':0.15}
    df['Risk_Score'] = sum(df[k]*v for k,v in weights.items())
    df['Risk_Level'] = pd.cut(df['Risk_Score'],bins=[0,3,6,8,10],labels=['Low','Moderate','High','Critical'],include_lowest=True)
    return df

# Main function
def run_employee_risk_analysis(file_path):
    df = load_data(file_path)
    df = preprocess_data(df)
    df = calculate_risk_dimensions(df)
    df = detect_anomalies(df)
    df = create_peer_comparisons(df)
    df = calculate_risk_score(df)
    df.to_csv('employee_risk_scores.csv', index=False)
    return df

if __name__ == "__main__":
    df = run_employee_risk_analysis(r"C:\Users\bhave\Downloads\GC-Opensoft-2025\emp_master.csv")
    print("Analysis complete. Results saved to employee_risk_scores.csv")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\bhave\AppData\Local\Programs\Python\Python312\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\bhave\AppData\Local\Programs\Python\Python312\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\bhave\AppData\Local\Programs\Python\Python312\Lib\site-packages\ipykernel\kernelapp.py", 

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\bhave\AppData\Local\Programs\Python\Python312\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\bhave\AppData\Local\Programs\Python\Python312\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\bhave\AppData\Local\Programs\Python\Python312\Lib\site-packages\ipykernel\kernelapp.py", 

AttributeError: _ARRAY_API not found

Analysis complete. Results saved to employee_risk_scores.csv


In [5]:
!pip install tensorflow

  Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl (15.5 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.4
    Uninstalling numpy-2.2.4:
      Successfully uninstalled numpy-2.2.4


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
nlopt 2.9.1 requires numpy<3,>=2, but you have numpy 1.26.4 which is incompatible.

[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.metrics import silhouette_score
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.models import Model
import warnings
warnings.filterwarnings('ignore')

class EmployeeRiskAnalyzer:
    def __init__(self):
        self.scaler = StandardScaler()
        self.pca = PCA(n_components=0.95)  # Preserve 95% variance
        self.anomaly_detector = IsolationForest(contamination=0.1, random_state=42)
        
    def engineer_features(self, df):
        """Advanced feature engineering with temporal and behavioral patterns"""
        
        # Communication patterns
        df['comm_diversity'] = df.apply(
            lambda x: entropy([
                x['Teams_Messages_Sent'],
                x['Emails_Sent'],
                x['Meetings_Attended']
            ]), axis=1
        )
        
        # Work-life balance indicators
        df['work_intensity'] = df['Work_Hours'] / (20 - df['Leave_Days'])
        df['engagement_consistency'] = df['Has_Response'] / df['Has_Activity']
        
        # Performance-effort alignment
        df['effort_score'] = (
            0.4 * df['Work_Hours'] +
            0.3 * df['Total_Communications'] +
            0.3 * df['Has_Activity']
        )
        df['performance_gap'] = df['effort_score'] - df['Performance_Rating']
        
        # Recognition metrics
        df['recognition_ratio'] = df['Has_Award'] / df['Performance_Rating']
        
        # Vibe score patterns
        df['vibe_volatility'] = df.groupby('Employee_ID')['Vibe_Score'].transform('std')
        
        return df
    
    def detect_anomalies(self, features):
        """Multi-dimensional anomaly detection using Isolation Forest and Autoencoder"""
        
        # Isolation Forest detection
        iso_scores = self.anomaly_detector.fit_predict(features)
        
        # Autoencoder for complex pattern detection
        input_dim = features.shape[1]
        input_layer = Input(shape=(input_dim,))
        encoded = Dense(int(input_dim/2), activation='relu')(input_layer)
        encoded = Dense(int(input_dim/4), activation='relu')(encoded)
        decoded = Dense(int(input_dim/2), activation='relu')(encoded)
        decoded = Dense(input_dim, activation='sigmoid')(decoded)
        
        autoencoder = Model(input_layer, decoded)
        autoencoder.compile(optimizer='adam', loss='mse')
        
        # Train autoencoder
        autoencoder.fit(features, features, epochs=50, batch_size=32, verbose=0)
        
        # Get reconstruction error
        reconstructed = autoencoder.predict(features)
        mse = np.mean(np.power(features - reconstructed, 2), axis=1)
        
        # Combine both scores
        final_anomaly_scores = (mse + (iso_scores == -1)) / 2
        return final_anomaly_scores
    
    def cluster_employees(self, features, n_clusters_range=range(2, 11)):
        """Dynamic clustering with optimal cluster selection"""
        
        # Find optimal number of clusters
        silhouette_scores = []
        for n_clusters in n_clusters_range:
            kmeans = KMeans(n_clusters=n_clusters, random_state=42)
            cluster_labels = kmeans.fit_predict(features)
            score = silhouette_score(features, cluster_labels)
            silhouette_scores.append(score)
        
        optimal_clusters = n_clusters_range[np.argmax(silhouette_scores)]
        
        # Perform final clustering
        kmeans = KMeans(n_clusters=optimal_clusters, random_state=42)
        return kmeans.fit_predict(features)
    
    def calculate_risk_weights(self, df, target_col='Performance_Rating'):
        """Calculate data-driven risk weights using Random Forest feature importance"""
        
        feature_cols = [
            'comm_diversity', 'work_intensity', 'engagement_consistency',
            'performance_gap', 'recognition_ratio', 'vibe_volatility'
        ]
        
        # Create binary target (low performers vs others)
        target = (df[target_col] < df[target_col].median()).astype(int)
        
        # Train Random Forest
        rf = RandomForestClassifier(n_estimators=100, random_state=42)
        rf.fit(df[feature_cols], target)
        
        # Get feature importance as weights
        weights = dict(zip(feature_cols, rf.feature_importances_))
        return weights
    
    def identify_at_risk_employees(self, df):
        """Main method to identify employees at risk"""
        
        # 1. Feature engineering
        df = self.engineer_features(df)
        
        # 2. Prepare features for analysis
        feature_cols = [
            'comm_diversity', 'work_intensity', 'engagement_consistency',
            'performance_gap', 'recognition_ratio', 'vibe_volatility'
        ]
        features = self.scaler.fit_transform(df[feature_cols])
        
        # 3. Dimensionality reduction
        reduced_features = self.pca.fit_transform(features)
        
        # 4. Anomaly detection
        anomaly_scores = self.detect_anomalies(reduced_features)
        
        # 5. Employee clustering
        cluster_labels = self.cluster_employees(reduced_features)
        
        # 6. Calculate risk weights
        risk_weights = self.calculate_risk_weights(df)
        
        # 7. Calculate final risk score
        df['risk_score'] = np.zeros(len(df))
        for feature, weight in risk_weights.items():
            df['risk_score'] += df[feature] * weight
        
        # 8. Normalize risk scores
        df['risk_score'] = (df['risk_score'] - df['risk_score'].min()) / (df['risk_score'].max() - df['risk_score'].min())
        
        # 9. Combine with anomaly scores
        df['final_risk_score'] = 0.7 * df['risk_score'] + 0.3 * anomaly_scores
        
        # 10. Categorize risk levels
        df['risk_level'] = pd.qcut(
            df['final_risk_score'],
            q=4,
            labels=['Low', 'Moderate', 'High', 'Critical']
        )
        
        return df

def entropy(values):
    """Calculate entropy of a distribution"""
    values = np.array(values) + 1e-10  # Avoid log(0)
    distribution = values / np.sum(values)
    return -np.sum(distribution * np.log2(distribution))

if __name__ == "__main__":
    # Example usage
    analyzer = EmployeeRiskAnalyzer()
    df = pd.read_csv("emp_master.csv")
    results = analyzer.identify_at_risk_employees(df)
    results.to_csv("employee_risk_analysis_results.csv", index=False)
    print("Analysis complete. Results saved to employee_risk_analysis_results.csv")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\bhave\AppData\Local\Programs\Python\Python312\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\bhave\AppData\Local\Programs\Python\Python312\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\bhave\AppData\Local\Programs\Python\Python312\Lib\site-packages\ipykernel\kernelapp.py", 

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\bhave\AppData\Local\Programs\Python\Python312\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\bhave\AppData\Local\Programs\Python\Python312\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\bhave\AppData\Local\Programs\Python\Python312\Lib\site-packages\ipykernel\kernelapp.py", 

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\bhave\AppData\Local\Programs\Python\Python312\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\bhave\AppData\Local\Programs\Python\Python312\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\bhave\AppData\Local\Programs\Python\Python312\Lib\site-packages\ipykernel\kernelapp.py", 

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\bhave\AppData\Local\Programs\Python\Python312\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\bhave\AppData\Local\Programs\Python\Python312\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\bhave\AppData\Local\Programs\Python\Python312\Lib\site-packages\ipykernel\kernelapp.py", 

AttributeError: _UFUNC_API not found

ImportError: numpy.core.umath failed to import